# Base model evaluation
In this notebook, the performance of selected base models will be evaluated on long texts.

**Selected models:**
- Encoders:
    - XLM-RoBERTa-large (FacebookAI/xlm-roberta-large) (0.6B parameters)
    - Qwen3-Embedding-0.6B (Qwen/Qwen3-Embedding-0.6B) (0.6B parameters)
- Decoders:
    - Qwen3-0.6B (Qwen/Qwen3-0.6B) (0.6B parameters)
    - Llama-3.2-1B (meta-llama/Llama-3.2-1B) (1B parameters)

In [1]:
!pip install mteb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 47.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 19.3 MB/s eta 0:00:00


In [2]:
import numpy as np

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from datasets.dataset_dict import DatasetDict
from datasets.arrow_dataset import Dataset
import datasets
import os

In [3]:
import mteb

In [4]:
tasks = mteb.get_tasks(
    languages=["eng"],
    exclude_aggregate=True,
    task_types=[
        "Classification",
        "Retrieval",
        "Summarization",
    ],
    modalities=["text"]
)

In [ ]:
datasets.utils.logging.disable_progress_bar()
os.environ["HF_DATASETS_DISABLE_CACHE"] = "1"


task_texts_length = {}
tasks_names = {}

for task in tasks:

    if task.languages != ["eng"]:
        continue

    task.load_data()
    dataset = task.dataset
    texts_len = []

    while not isinstance(dataset, Dataset):
        dataset = list(dataset.values())[0]

    column = "text"
    if not column in dataset.features:
        column = "sentences"

    for text in dataset[column]:
        texts_len.append(len(text))
    tasks_names[task.metadata.name] = task
    task_texts_length[task.metadata.name] = {
        "min_len": min(texts_len),
        "max_len": max(texts_len),
        "mean_len": np.mean(texts_len)
    }

## Baseline models evaluation

In [5]:
def chunk_text(text, tokenizer, chunk_size) -> list:
    tokens = tokenizer(text, return_tensors=None, add_special_tokens=False)

    chunks = []
    for i in range(0, len(tokens), chunk_size):
        chunk = tokens[i:i + max_tokens]
        chunks.append(chunk)
    return chunks

In [9]:
from transformers import AutoTokenizer, AutoModel

"""Classes used to implement an encode funciton
passed to mteb.evaluate() funciton to evaluate on tasks
"""

class Quen3Embedding:

    name = "Qwen3-Embedding-0.6B"

    def __init__(self, chunk_size):
        self.chunk_size = chunk_size
        self.model = AutoModel.from_pretrained("Qwen/Qwen3-Embedding-0.6B")
        self.tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-Embedding-0.6B")

    def encode(
            self,
            inputs,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type = None,
            **kwargs) -> torch.tensor:
        """Encodes the given sentences using the encoder.

        Args:
            inputs: The inputs to encode.
            task_metadata: The name of the task.
            hf_subset: The subset of the dataset.
            hf_split: The split of the dataset.
            prompt_type: The prompt type to use.
            **kwargs: Additional arguments to pass to the encoder.

        Returns:
            The encoded sentences.
        """

        embeddings = []

        chunk_tokenized = chunk_text(inputs, self.tokenizer, self.chunk_size)
        for chunk in chunk_tokenized:
            ouputs = self.model(chunk)
            embeddings.append(outputs.last_hidden_state)

        final_embedding = embeddings.average()
        return final_embeddin


class XMLRoBERTa:

    name = "xlm-roberta-large"

    def __init__(self, chunk_size):
        self.chunk_size = chunk_size
        self.model = AutoModel.from_pretrained("FacebookAI/xlm-roberta-large")
        self.tokenizer = AutoTokenizer.from_pretrained("FacebookAI/xlm-roberta-large")

    def encode(
            self,
            inputs,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type = None,
            **kwargs) -> torch.tensor:
        embeddings = []

        chunk_tokenized = chunk_text(inputs, self.tokenizer, self.chunk_size)
        for chunk in chunk_tokenized:
            ouputs = self.model(chunk)
            embeddings.append(outputs.last_hidden_state)

        final_embedding = embeddings.average()
        return final_embedding


class Qwen3:

    name = "Qwen3-0.6B"

    def __init__(self, chunk_size):
        self.chunk_size = chunk_size
        self.model = AutoModel.from_pretrained("Qwen/Qwen3-0.6B")
        self.tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

    def encode(
            self,
            inputs,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type = None,
            **kwargs) -> torch.tensor:
        embeddings = []

        chunk_tokenized = chunk_text(inputs, self.tokenizer, self.chunk_size)
        for chunk in chunk_tokenized:
            ouputs = self.model(chunk)
            embeddings.append(outputs.last_hidden_state)

        final_embedding = embeddings.average()
        return final_embedding

class Llama3_2:

    name = "Llama-3.2-1B"

    def __init__(self, chunk_size):
        self.chunk_size = chunk_size
        self.model = AutoModel.from_pretrained("meta-llama/Llama-3.2-1B")
        self.tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")

    def encode(
            self,
            inputs,
            task_metadata,
            hf_split: str,
            hf_subset: str,
            prompt_type = None,
            **kwargs) -> torch.tensor:
        embeddings = []

        chunk_tokenized = chunk_text(inputs, self.tokenizer, self.chunk_size)
        for chunk in chunk_tokenized:
            ouputs = self.model(chunk)
            embeddings.append(outputs.last_hidden_state)

        final_embedding = embeddings.average()
        return final_embedding

### Testing on example data

In [40]:
example = mteb.get_task("QMSum")
example.load_data()
data = example.dataset

KeyError: "KeyError: 'QMSum' not found. Did you mean: 'SAMSumFa'?"

In [39]:
data.items()

dict_items([('train', defaultdict(<function AbsTaskRetrieval.__init__.<locals>.<lambda>.<locals>.<lambda> at 0x7ca5b0bfa520>, {7: {'corpus': Dataset({
    features: [],
    num_rows: 0
}), 'queries': Dataset({
    features: [],
    num_rows: 0
}), 'relevant_docs': {}, 'top_ranked': None}, 0: {'corpus': Dataset({
    features: [],
    num_rows: 0
}), 'queries': Dataset({
    features: [],
    num_rows: 0
}), 'relevant_docs': {}, 'top_ranked': None}}))])

In [37]:
train_shard = data['train'][0]

corpus_dataset = train_shard['corpus']
query_dataset = train_shard['queries']
relevant_docs = train_shard['relevant_docs']

print(corpus_dataset.features)
print(len(corpus_dataset))
print(corpus_dataset[0]['text'])

{}
0


IndexError: Invalid key: 0 is out of bounds for size 0

In [36]:
data

defaultdict(<function mteb.abstasks.retrieval.AbsTaskRetrieval.__init__.<locals>.<lambda>()>,
            {'train': defaultdict(<function mteb.abstasks.retrieval.AbsTaskRetrieval.__init__.<locals>.<lambda>.<locals>.<lambda>()>,
                         {7: {'corpus': Dataset({
                               features: [],
                               num_rows: 0
                           }),
                           'queries': Dataset({
                               features: [],
                               num_rows: 0
                           }),
                           'relevant_docs': {},
                           'top_ranked': None}})})